# Analytical Query Library — Tasks 1 and 2

This notebook defines the analytical requirements and implements them with Spark SQL against the Week 1 Delta tables.

## Scope and common definitions

- **Taxi demand:** number of valid taxi trips, counted with `COUNT(trip_id)`.
- **Taxi zone:** pickup taxi zone; demand is attributed to where a trip starts.
- **Month:** calendar month in the `America/New_York` session time zone.
- **Weather condition:** Meteostat `coco` / integrated `weather_condition_code`, grouped into clear/fair, cloudy, fog, rain, snow/sleet, storm, and unknown.
- **Air quality:** borough-hour mean PM2.5 (`air_quality_pm25_mean`). Demand is first aggregated to the same borough-hour grain so repeated trip-level PM2.5 values do not bias the analysis.
- **Peak hour:** hour(s) with the highest trip count for a day of week; ties are retained.
- Missing weather or air-quality matches are excluded only from analyses that require that context.

The queries return analytical result sets only; they do not write data products. Materialized Delta outputs belong to Task 4.

In [ ]:
from pathlib import Path

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

builder = (
    SparkSession.builder
    .appName("Lab2-Analytical-Queries")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "America/New_York")
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Processed data:", PROCESSED_DIR)

In [ ]:
DELTA_TABLES = {
    "taxi_trips": PROCESSED_DIR / "taxi_trips",
    "weather": PROCESSED_DIR / "weather",
    "air_quality": PROCESSED_DIR / "air_quality",
    "taxi_zones": PROCESSED_DIR / "taxi_zones",
    "integrated_taxi_trips": PROCESSED_DIR / "integrated_taxi_trips",
}

missing = [name for name, path in DELTA_TABLES.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing Delta tables: {missing}. Run the Week 1 ingestion and integration notebooks first."
    )

for view_name, path in DELTA_TABLES.items():
    spark.read.format("delta").load(str(path)).createOrReplaceTempView(view_name)
    print(f"Registered {view_name}: {path}")

QUERIES = {}

## Task 1 — Analytical query design

1. **Monthly taxi demand for each taxi zone**
   - Grain: pickup year × pickup month × pickup zone.
   - Measure: trip count. Zone attributes come from the lookup Delta table.

2. **Average trip distance under different weather conditions**
   - Grain: weather category.
   - Measures: average distance, standard deviation, and trip count.
   - Only exact weather matches are included.

3. **Relationship between air quality and taxi demand**
   - First aggregate trips to borough-hour, matching the grain of PM2.5.
   - Report average hourly demand by PM2.5 band and Pearson correlation between PM2.5 and hourly demand.
   - Only exact borough-hour air-quality matches are included. Results describe association, not causation.

4. **Taxi zones with the largest variation in demand under different weather conditions**
   - Construct every zone × observed weather-hour combination so zero-demand hours are retained.
   - Compute average hourly zone demand per weather category, then rank zones by the range and population standard deviation across categories.
   - Normalizing to hourly demand prevents common weather categories from appearing more influential merely because they occur more often.

5. **Peak travel hours for each day of the week**
   - Count trips by day of week × hour of day, then use `DENSE_RANK` so tied peaks are retained.

6. **Monthly trends in taxi demand**
   - Report monthly trip count, average daily demand, month-over-month absolute change, and percentage change.

In [ ]:
# Reusable SQL view for the Meteostat condition-code grouping used by Query 4.
spark.sql("""
CREATE OR REPLACE TEMP VIEW weather_hours AS
SELECT
    observation_timestamp AS weather_hour,
    CASE
        WHEN coco IN (1, 2) THEN 'clear_or_fair'
        WHEN coco IN (3, 4) THEN 'cloudy'
        WHEN coco IN (5, 6) THEN 'fog'
        WHEN coco IN (7, 8, 9, 10, 11, 17, 18) THEN 'rain'
        WHEN coco IN (12, 13, 14, 15, 16, 19, 20, 21, 22) THEN 'snow_or_sleet'
        WHEN coco BETWEEN 23 AND 27 THEN 'storm'
        ELSE 'unknown'
    END AS weather_condition
FROM weather
WHERE observation_timestamp IS NOT NULL
""")

spark.sql("SELECT weather_condition, COUNT(*) AS observed_hours FROM weather_hours GROUP BY weather_condition ORDER BY weather_condition").show(truncate=False)

## Task 2 — Spark SQL implementation

### Query 1: Monthly taxi demand for each pickup zone

This implementation intentionally reads the underlying trip and zone Delta tables. `pickup_year` and `pickup_month` are retained as separate columns so later experiments can filter their physical partitions directly.

In [ ]:
QUERIES["monthly_zone_demand"] = """
SELECT
    t.pickup_year,
    t.pickup_month,
    t.pickup_location_id,
    COALESCE(z.borough, 'Unknown') AS pickup_borough,
    COALESCE(z.zone, 'Unknown') AS pickup_zone,
    COUNT(t.trip_id) AS trip_count
FROM taxi_trips AS t
LEFT JOIN taxi_zones AS z
    ON t.pickup_location_id = z.location_id
GROUP BY
    t.pickup_year,
    t.pickup_month,
    t.pickup_location_id,
    COALESCE(z.borough, 'Unknown'),
    COALESCE(z.zone, 'Unknown')
ORDER BY t.pickup_year, t.pickup_month, trip_count DESC, t.pickup_location_id
"""

spark.sql(QUERIES["monthly_zone_demand"]).show(50, truncate=False)

### Query 2: Average trip distance under different weather conditions

The integrated Delta table already contains the exact pickup-hour weather match. The result includes dispersion and sample size so the mean can be interpreted responsibly.

In [ ]:
QUERIES["distance_by_weather"] = """
WITH classified AS (
    SELECT
        trip_id,
        trip_distance,
        CASE
            WHEN weather_condition_code IN (1, 2) THEN 'clear_or_fair'
            WHEN weather_condition_code IN (3, 4) THEN 'cloudy'
            WHEN weather_condition_code IN (5, 6) THEN 'fog'
            WHEN weather_condition_code IN (7, 8, 9, 10, 11, 17, 18) THEN 'rain'
            WHEN weather_condition_code IN (12, 13, 14, 15, 16, 19, 20, 21, 22) THEN 'snow_or_sleet'
            WHEN weather_condition_code BETWEEN 23 AND 27 THEN 'storm'
            ELSE 'unknown'
        END AS weather_condition
    FROM integrated_taxi_trips
    WHERE weather_match_status = 'exact_hour'
      AND trip_distance IS NOT NULL
)
SELECT
    weather_condition,
    COUNT(trip_id) AS trip_count,
    ROUND(AVG(trip_distance), 3) AS average_trip_distance,
    ROUND(STDDEV_POP(trip_distance), 3) AS trip_distance_stddev
FROM classified
GROUP BY weather_condition
ORDER BY average_trip_distance DESC
"""

spark.sql(QUERIES["distance_by_weather"]).show(truncate=False)

### Query 3: Relationship between air quality and taxi demand

A trip-level average would overweight PM2.5 hours that already have many trips. Both outputs therefore build one record per borough-hour first. The band summary is easy to report; Pearson correlation provides a compact measure of linear association.

In [ ]:
QUERIES["air_quality_demand_by_band"] = """
WITH borough_hour AS (
    SELECT
        pickup_borough,
        pickup_hour,
        AVG(air_quality_pm25_mean) AS pm25,
        COUNT(trip_id) AS hourly_trip_count
    FROM integrated_taxi_trips
    WHERE air_quality_match_status = 'borough_exact_hour'
      AND pickup_borough IS NOT NULL
      AND air_quality_pm25_mean IS NOT NULL
    GROUP BY pickup_borough, pickup_hour
), banded AS (
    SELECT
        *,
        CASE
            WHEN pm25 <= 5.0 THEN '01_very_low_0_5'
            WHEN pm25 <= 15.0 THEN '02_low_5_15'
            WHEN pm25 <= 35.0 THEN '03_elevated_15_35'
            ELSE '04_high_over_35'
        END AS pm25_band
    FROM borough_hour
)
SELECT
    pm25_band,
    COUNT(*) AS borough_hour_count,
    ROUND(AVG(pm25), 3) AS average_pm25,
    ROUND(AVG(hourly_trip_count), 2) AS average_hourly_demand,
    MIN(hourly_trip_count) AS minimum_hourly_demand,
    MAX(hourly_trip_count) AS maximum_hourly_demand
FROM banded
GROUP BY pm25_band
ORDER BY pm25_band
"""

QUERIES["air_quality_demand_correlation"] = """
WITH borough_hour AS (
    SELECT
        pickup_borough,
        pickup_hour,
        AVG(air_quality_pm25_mean) AS pm25,
        COUNT(trip_id) AS hourly_trip_count
    FROM integrated_taxi_trips
    WHERE air_quality_match_status = 'borough_exact_hour'
      AND pickup_borough IS NOT NULL
      AND air_quality_pm25_mean IS NOT NULL
    GROUP BY pickup_borough, pickup_hour
)
SELECT
    COALESCE(pickup_borough, 'All boroughs') AS pickup_borough,
    COUNT(*) AS matched_borough_hours,
    ROUND(CORR(pm25, hourly_trip_count), 4) AS pm25_demand_pearson_correlation
FROM borough_hour
GROUP BY GROUPING SETS ((pickup_borough), ())
ORDER BY pickup_borough
"""

spark.sql(QUERIES["air_quality_demand_by_band"]).show(truncate=False)
spark.sql(QUERIES["air_quality_demand_correlation"]).show(truncate=False)

### Query 4: Zones with the largest weather-related demand variation

This query uses the underlying tables to include zero-trip zone-hours. A zone must have observations in at least two weather categories to be ranked.

In [ ]:
QUERIES["zone_weather_demand_variation"] = """
WITH trip_bounds AS (
    SELECT MIN(pickup_hour) AS first_hour, MAX(pickup_hour) AS last_hour
    FROM taxi_trips
), relevant_weather_hours AS (
    SELECT w.weather_hour, w.weather_condition
    FROM weather_hours AS w
    CROSS JOIN trip_bounds AS b
    WHERE w.weather_hour BETWEEN b.first_hour AND b.last_hour
      AND w.weather_condition <> 'unknown'
), zone_hour_counts AS (
    SELECT
        pickup_location_id AS location_id,
        pickup_hour,
        COUNT(trip_id) AS trip_count
    FROM taxi_trips
    GROUP BY pickup_location_id, pickup_hour
), zone_weather_hour_grid AS (
    SELECT
        z.location_id,
        z.borough,
        z.zone,
        w.weather_condition,
        w.weather_hour,
        COALESCE(c.trip_count, 0) AS trip_count
    FROM taxi_zones AS z
    CROSS JOIN relevant_weather_hours AS w
    LEFT JOIN zone_hour_counts AS c
        ON z.location_id = c.location_id
       AND w.weather_hour = c.pickup_hour
), zone_weather_average AS (
    SELECT
        location_id,
        borough,
        zone,
        weather_condition,
        AVG(trip_count) AS average_hourly_demand
    FROM zone_weather_hour_grid
    GROUP BY location_id, borough, zone, weather_condition
), variation AS (
    SELECT
        location_id,
        borough,
        zone,
        COUNT(*) AS weather_categories,
        MIN(average_hourly_demand) AS minimum_weather_demand,
        MAX(average_hourly_demand) AS maximum_weather_demand,
        MAX(average_hourly_demand) - MIN(average_hourly_demand) AS demand_range,
        STDDEV_POP(average_hourly_demand) AS demand_stddev
    FROM zone_weather_average
    GROUP BY location_id, borough, zone
    HAVING COUNT(*) >= 2
), ranked AS (
    SELECT
        *,
        DENSE_RANK() OVER (ORDER BY demand_range DESC) AS variation_rank
    FROM variation
)
SELECT
    variation_rank,
    location_id,
    borough,
    zone,
    weather_categories,
    ROUND(minimum_weather_demand, 3) AS minimum_average_hourly_demand,
    ROUND(maximum_weather_demand, 3) AS maximum_average_hourly_demand,
    ROUND(demand_range, 3) AS average_hourly_demand_range,
    ROUND(demand_stddev, 3) AS average_hourly_demand_stddev
FROM ranked
WHERE variation_rank <= 20
ORDER BY variation_rank, location_id
"""

spark.sql(QUERIES["zone_weather_demand_variation"]).show(50, truncate=False)

### Query 5: Peak travel hours for each day of the week

In [ ]:
QUERIES["peak_hours_by_day_of_week"] = """
WITH day_hour_demand AS (
    SELECT
        DAYOFWEEK(pickup_datetime) AS day_of_week_number,
        DATE_FORMAT(pickup_datetime, 'EEEE') AS day_of_week,
        HOUR(pickup_datetime) AS hour_of_day,
        COUNT(trip_id) AS trip_count
    FROM taxi_trips
    GROUP BY
        DAYOFWEEK(pickup_datetime),
        DATE_FORMAT(pickup_datetime, 'EEEE'),
        HOUR(pickup_datetime)
), ranked AS (
    SELECT
        *,
        DENSE_RANK() OVER (
            PARTITION BY day_of_week_number
            ORDER BY trip_count DESC
        ) AS demand_rank
    FROM day_hour_demand
)
SELECT
    day_of_week_number,
    day_of_week,
    hour_of_day AS peak_hour,
    trip_count
FROM ranked
WHERE demand_rank = 1
ORDER BY day_of_week_number, peak_hour
"""

spark.sql(QUERIES["peak_hours_by_day_of_week"]).show(truncate=False)

### Query 6: Monthly trends in taxi demand

Average daily demand is included because raw monthly totals are affected by the number of calendar days represented.

In [ ]:
QUERIES["monthly_demand_trends"] = """
WITH monthly AS (
    SELECT
        pickup_year,
        pickup_month,
        MAKE_DATE(pickup_year, pickup_month, 1) AS month_start,
        COUNT(trip_id) AS trip_count,
        COUNT(DISTINCT pickup_date) AS observed_days,
        COUNT(trip_id) / COUNT(DISTINCT pickup_date) AS average_daily_demand
    FROM taxi_trips
    GROUP BY pickup_year, pickup_month
), with_previous AS (
    SELECT
        *,
        LAG(trip_count) OVER (ORDER BY month_start) AS previous_month_trip_count
    FROM monthly
)
SELECT
    pickup_year,
    pickup_month,
    month_start,
    trip_count,
    observed_days,
    ROUND(average_daily_demand, 2) AS average_daily_demand,
    trip_count - previous_month_trip_count AS month_over_month_change,
    ROUND(
        100.0 * (trip_count - previous_month_trip_count)
        / NULLIF(previous_month_trip_count, 0),
        2
    ) AS month_over_month_change_pct
FROM with_previous
ORDER BY month_start
"""

spark.sql(QUERIES["monthly_demand_trends"]).show(50, truncate=False)

## Completion checks

Running the notebook from top to bottom registers five Week 1 Delta tables and executes all six required analyses. Query 3 intentionally has two complementary result sets, so the library contains seven SQL statements in total.

The output of these cells is the Task 2 baseline. Task 3 should benchmark equivalent optimized variants against these results without changing the analytical definitions above.

In [ ]:
EXPECTED_QUERIES = {
    "monthly_zone_demand",
    "distance_by_weather",
    "air_quality_demand_by_band",
    "air_quality_demand_correlation",
    "zone_weather_demand_variation",
    "peak_hours_by_day_of_week",
    "monthly_demand_trends",
}
assert set(QUERIES) == EXPECTED_QUERIES
print(f"Analytical query library complete: {len(QUERIES)} SQL statements")
for query_name in QUERIES:
    print(" -", query_name)